In [1]:
import ee
import pandas as pd
from datetime import datetime, timedelta

import geemap
# geemap.set_proxy(port=56370)
Map=geemap.Map()#设置地图大小
Map
# 初始化GEE
ee.Initialize()

# collection

In [21]:
# 配置参数
TIME_WINDOW_HOURS = 10  # 时间窗口(小时)
MIN_OVERLAP_RATIO = 0.5  # 最小重叠比例(30%)
# 定义数据集配置
DATASETS = {
    'Landsat8_TOA': 'LANDSAT/LC08/C02/T1_TOA',
    'Landsat8_RT': 'LANDSAT/LC08/C02/T1_RT',
    'Landsat7_RT': 'LANDSAT/LE07/C02/T1_RT',
    'Landsat5_T1_L2': 'LANDSAT/LT05/C02/T1_L2',
    'Sentinel2': 'COPERNICUS/S2_HARMONIZED',
    'Sentinel2_SR': 'COPERNICUS/S2_SR_HARMONIZED'

}

def parse_s1_filename(scene_name):
    """解析Sentinel-1文件名获取时间"""
    parts = scene_name.split('_')
    date_str = parts[4]  # 例如: 20150107T215431
    dt = datetime.strptime(date_str, '%Y%m%dT%H%M%S')
    return dt

def get_s1_geometry(scene_name):
    """从GEE获取Sentinel-1影像的几何信息"""
    try:
        s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD')
        img = s1_collection.filter(ee.Filter.eq('system:index', scene_name)).first()
        return img.geometry()
    except:
        return None

def calculate_overlap_ratio(geom1, geom2):
    """计算两个几何体的重叠比例"""
    try:
        intersection = geom1.intersection(geom2, ee.ErrorMargin(1))
        intersection_area = intersection.area(ee.ErrorMargin(1))
        geom1_area = geom1.area(ee.ErrorMargin(1))
        overlap_ratio = intersection_area.divide(geom1_area)
        return overlap_ratio.getInfo()
    except:
        return 0

def search_collection(collection_name, dataset_label, start_str, end_str, s1_geom, s1_scene, s1_time):
    """搜索指定数据集的重叠影像"""
    results = []
    try:
        collection = ee.ImageCollection(collection_name) \
            .filterDate(start_str, end_str) \
            .filterBounds(s1_geom)
        
        img_list = collection.toList(1000)
        img_size = img_list.size().getInfo()
        
        for i in range(img_size):
            img = ee.Image(img_list.get(i))
            img_geom = img.geometry()
            overlap = calculate_overlap_ratio(s1_geom, img_geom)
            
            if overlap >= MIN_OVERLAP_RATIO:
                img_id = img.get('system:index').getInfo()
                img_time = datetime.fromtimestamp(img.get('system:time_start').getInfo() / 1000)
                time_diff = abs((img_time - s1_time).total_seconds() / 3600)
                
                results.append({
                    's1_scene': s1_scene,
                    'sensor': dataset_label,
                    'collection': collection_name,
                    'image_id': img_id,
                    'overlap_ratio': round(overlap, 3),
                    'time_diff_hours': round(time_diff, 2),
                    's1_time': s1_time.strftime('%Y-%m-%d %H:%M:%S'),
                    'matched_time': img_time.strftime('%Y-%m-%d %H:%M:%S')
                })
    except Exception as e:
        print(f"  {dataset_label} 查询错误: {type(e).__name__}: {str(e)}")
    
    return results

def find_overlapping_images(s1_scene, s1_time, s1_geom, time_window_hours):
    """查找所有数据集中与Sentinel-1重叠的影像"""
    results = []
    
    # 计算时间范围
    start_time = s1_time - timedelta(hours=time_window_hours)
    end_time = s1_time + timedelta(hours=time_window_hours)
    
    start_str = start_time.isoformat()
    end_str = end_time.isoformat()
    
    if s1_geom is None:
        return results
    
    # 遍历所有数据集
    for dataset_label, collection_name in DATASETS.items():
        matches = search_collection(collection_name, dataset_label, start_str, end_str, 
                                    s1_geom, s1_scene, s1_time)
        results.extend(matches)
        if matches:
            print(f"  {dataset_label}: {len(matches)} 个")
    
    return results

In [ ]:
if __name__ == "__main__":
    # 读取CSV文件
    input_csv = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\statistic_20251231\overall_density_summary_2023_raster_filtered_detailed.csv"
    df = pd.read_csv(input_csv)
    
    all_results = []
    
    # 遍历每个Sentinel-1场景
    for idx, row in df.iterrows():
        s1_scene = row['scene_name']
        print(f"处理 {idx+1}/{len(df)}: {s1_scene}")
        
        try:
            # 解析时间
            s1_time = parse_s1_filename(s1_scene)
            
            # 获取几何信息
            s1_geom = get_s1_geometry(s1_scene)
            
            # 查找重叠影像
            matches = find_overlapping_images(s1_scene, s1_time, s1_geom, TIME_WINDOW_HOURS)
            all_results.extend(matches)
            
            if matches:
                print(f"  总计: {len(matches)} 个匹配影像")
            else:
                print(f"  无匹配影像")
            
        except Exception as e:
            print(f"  错误: {e}")
    
    # 保存结果
    if all_results:
        result_df = pd.DataFrame(all_results)
        output_csv = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\overlapping_S1_Optical.csv"
        result_df.to_csv(output_csv, index=False)
        
        print(f"\n{'='*60}")
        print(f"完成! 共找到 {len(all_results)} 个重叠影像")
        print(f"结果已保存到: {output_csv}")
        
        # 统计各传感器数量
        print(f"\n各传感器统计:")
        sensor_counts = result_df['sensor'].value_counts()
        for sensor, count in sensor_counts.items():
            print(f"  {sensor}: {count} 个")
        
        print(f"\n前5条结果预览:")
        print(result_df.head().to_string())
    else:
        print("\n未找到任何重叠影像")

处理 1/51: S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83


# single

In [2]:

# 配置参数
TIME_WINDOW_HOURS = 6  # 时间窗口(小时)
MIN_OVERLAP_RATIO = 0.5  # 最小重叠比例(30%)

def parse_s1_filename(scene_name):
    """解析Sentinel-1文件名获取时间和几何信息"""
    parts = scene_name.split('_')
    date_str = parts[4]  # 例如: 20150107T215431
    dt = datetime.strptime(date_str, '%Y%m%dT%H%M%S')
    return dt

def get_s1_geometry(scene_name):
    """从GEE获取Sentinel-1影像的几何信息"""
    try:
        s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD')
        img = s1_collection.filter(ee.Filter.eq('system:index', scene_name)).first()
        return img.geometry()
    except:
        # 如果无法直接获取，返回None
        return None

def calculate_overlap_ratio(geom1, geom2):
    """计算两个几何体的重叠比例"""
    try:
        intersection = geom1.intersection(geom2, ee.ErrorMargin(1))
        intersection_area = intersection.area(ee.ErrorMargin(1))
        geom1_area = geom1.area(ee.ErrorMargin(1))
        overlap_ratio = intersection_area.divide(geom1_area)
        return overlap_ratio.getInfo()
    except:
        return 0

def find_overlapping_images(s1_scene, s1_time, s1_geom, time_window_hours):
    """查找与Sentinel-1重叠的Landsat 8和Sentinel-2影像"""
    results = []
    
    # 计算时间范围
    start_time = s1_time - timedelta(hours=time_window_hours)
    end_time = s1_time + timedelta(hours=time_window_hours)
    
    start_str = start_time.strftime('%Y-%m-%d %H:%M:%S')
    end_str = end_time.strftime('%Y-%m-%d %H:%M:%S')
    
    if s1_geom is None:
        return results
    
    # 查找Landsat 8影像
    try:
        l8_collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
            .filterDate(start_str, end_str) \
            .filterBounds(s1_geom)
        
        l8_list = l8_collection.toList(1000)
        l8_size = l8_list.size().getInfo()
        
        for i in range(l8_size):
            img = ee.Image(l8_list.get(i))
            img_geom = img.geometry()
            overlap = calculate_overlap_ratio(s1_geom, img_geom)
            
            if overlap >= MIN_OVERLAP_RATIO:
                img_id = img.get('system:index').getInfo()
                img_time = datetime.fromtimestamp(img.get('system:time_start').getInfo() / 1000)
                time_diff = abs((img_time - s1_time).total_seconds() / 3600)
                
                results.append({
                    's1_scene': s1_scene,
                    'matched_sensor': 'Landsat8',
                    'matched_id': img_id,
                    'overlap_ratio': round(overlap, 3),
                    'time_diff_hours': round(time_diff, 2),
                    's1_time': s1_time.strftime('%Y-%m-%d %H:%M:%S'),
                    'matched_time': img_time.strftime('%Y-%m-%d %H:%M:%S')
                })
    except Exception as e:
        print(f"Landsat 8查询错误 ({s1_scene}): {e}")
    
    # 查找Sentinel-2影像
    try:
        s2_collection = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
            .filterDate(start_str, end_str) \
            .filterBounds(s1_geom)
        
        s2_list = s2_collection.toList(1000)
        s2_size = s2_list.size().getInfo()
        
        for i in range(s2_size):
            img = ee.Image(s2_list.get(i))
            img_geom = img.geometry()
            overlap = calculate_overlap_ratio(s1_geom, img_geom)
            
            if overlap >= MIN_OVERLAP_RATIO:
                img_id = img.get('system:index').getInfo()
                img_time = datetime.fromtimestamp(img.get('system:time_start').getInfo() / 1000)
                time_diff = abs((img_time - s1_time).total_seconds() / 3600)
                
                results.append({
                    's1_scene': s1_scene,
                    'matched_sensor': 'Sentinel2',
                    'matched_id': img_id,
                    'overlap_ratio': round(overlap, 3),
                    'time_diff_hours': round(time_diff, 2),
                    's1_time': s1_time.strftime('%Y-%m-%d %H:%M:%S'),
                    'matched_time': img_time.strftime('%Y-%m-%d %H:%M:%S')
                })
    except Exception as e:
        print(f"Sentinel-2查询错误 ({s1_scene}): {e}")
    
    return results



In [3]:


# 主程序
if __name__ == "__main__":
    # 读取CSV文件
    # input_csv = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\overlap_S1name.csv"  # 修改为你的文件路径
    input_csv = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\statistic_20251231\overall_density_summary_2023_raster_filtered_detailed.csv"
    df = pd.read_csv(input_csv)
    
    all_results = []
    
    # 遍历每个Sentinel-1场景
    for idx, row in df.iterrows():
        s1_scene = row['scene_name']
        print(f"处理 {idx+1}/{len(df)}: {s1_scene}")
        
        try:
            # 解析时间
            s1_time = parse_s1_filename(s1_scene)
            
            # 获取几何信息
            s1_geom = get_s1_geometry(s1_scene)
            
            # 查找重叠影像
            matches = find_overlapping_images(s1_scene, s1_time, s1_geom, TIME_WINDOW_HOURS)
            all_results.extend(matches)
            
            print(f"  找到 {len(matches)} 个匹配影像")
            
        except Exception as e:
            print(f"  错误: {e}")
    
    # 保存结果
    if all_results:
        result_df = pd.DataFrame(all_results)
        output_csv = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\overlapping_S1_Optical.csv"
        result_df.to_csv(output_csv, index=False)
        print(f"\n完成! 共找到 {len(all_results)} 个重叠影像")
        print(f"结果已保存到: {output_csv}")
        print(f"\n前5条结果预览:")
        print(result_df.head())
    else:
        print("\n未找到任何重叠影像")

处理 1/51: S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83
Landsat 8查询错误 (S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83): DateRange: Bad date/time '2023-06-30 07:47:49'.
Sentinel-2查询错误 (S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83): DateRange: Bad date/time '2023-06-30 07:47:49'.
  找到 0 个匹配影像
处理 2/51: S1A_EW_GRDM_1SDH_20230626T142248_20230626T142348_049158_05E944_BA88
Landsat 8查询错误 (S1A_EW_GRDM_1SDH_20230626T142248_20230626T142348_049158_05E944_BA88): DateRange: Bad date/time '2023-06-26 08:22:48'.
Sentinel-2查询错误 (S1A_EW_GRDM_1SDH_20230626T142248_20230626T142348_049158_05E944_BA88): DateRange: Bad date/time '2023-06-26 08:22:48'.
  找到 0 个匹配影像
处理 3/51: S1A_EW_GRDM_1SDH_20230626T142148_20230626T142248_049158_05E944_ECF8
Landsat 8查询错误 (S1A_EW_GRDM_1SDH_20230626T142148_20230626T142248_049158_05E944_ECF8): DateRange: Bad date/time '2023-06-26 08:21:48'.
Sentinel-2查询错误 (S1A_EW_GRDM_1SDH_20230626T142148_20230626T142248_049158_05E9